# Dev Environment → The Git Commands You Actually Use

---

## 🏗️ Build | 🚢 Ship | 📤 Share

### 🏗️ Build
The six commands that cover almost every change you'll ship: `clone`, `remote add upstream`, `pull`, `switch -c`, `add`/`commit`, and `push`.

### 🚢 Ship
Run the loop end to end today: pull from the course repo, make a branch, change one thing, and push it to your own repo.

### 📤 Share
Which step do you let your AI editor (Claude Code, Cursor) drive, and which one do you still want to run yourself?

---

You work on **your own copy** of the course repo and pull updates from ours. Git tracks both as *remotes*:

- `origin` → your repo. You **push** here.
- `upstream` → the course repo. You **pull** from here.

```
git pull upstream main   →   make changes   →   git add .   →   git commit   →   git push origin main
```

**Estimated time:** 10–15 minutes

## Setup

Every cell below runs a real `git` command through a small `sh()` helper that prints the command and its output, so you always see exactly what ran.

In [1]:
import subprocess


def sh(cmd, check=True, quiet=False):
    """Run a git command, print it with its output, and return stdout."""
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet:
        body = ((r.stdout or "") + (r.stderr or "")).rstrip()
        print(f"$ {cmd}\n{body}\n")
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return (r.stdout or "").strip()


sh("git --version")
print("repo root:", sh("git rev-parse --show-toplevel", quiet=True))

$ git --version
git version 2.46.0

repo root: /Users/katerinag/The-Hive-FDE-Challenge


## 1. `clone` — get the code onto your machine

You only do this once per repo. Create an empty repository on GitHub.com (leave *Add a README file* **deselected**), copy its SSH address, then run:

```bash
git clone git@github.com:YOUR_USERNAME/YOUR_REPO.git
cd YOUR_REPO
```

> `warning: You appear to have cloned an empty repository.` is expected on a brand new repo — it means it worked.

There's no cell for this one because you're already inside a clone. Everything below runs against it.

## 2. `remote add upstream` — point at the course repo

Your clone already knows about `origin` (your repo). Add the course repo as `upstream` so you can pull our updates.

In [2]:
# Change this if your instructor uses a different repo.
UPSTREAM_URL = "git@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git"

if "upstream" not in sh("git remote", quiet=True).split():
    sh(f"git remote add upstream {UPSTREAM_URL}")

sh("git remote -v")  # verify: you should see both origin and upstream

$ git remote add upstream git@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git


$ git remote -v
origin	git@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (fetch)
origin	git@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (push)
upstream	git@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (fetch)
upstream	git@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (push)



'origin\tgit@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (fetch)\norigin\tgit@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (push)\nupstream\tgit@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (fetch)\nupstream\tgit@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git (push)'

## 3. `pull` — get the latest course code

Start every session here so you're not building on stale code.

> On your very first pull into a fresh repo, add `--allow-unrelated-histories` — your empty repo and the course repo don't share any history yet.

In [3]:
sh("git pull upstream main", check=False)

# First pull into a brand new repo? Use this instead:
# sh("git pull upstream main --allow-unrelated-histories")

$ git pull upstream main
Already up to date.
From github.com:AI-Maker-Space/The-Hive-FDE-Challenge
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> upstream/main



'Already up to date.'

## 4. `switch -c` — make a branch for today's work

Keep `main` clean and do your work on a branch, named so a reviewer can tell what it's about at a glance.

- `git switch -c <name>` creates a branch and moves you onto it.
- `git switch <name>` moves to a branch that already exists.

In [4]:
BRANCH = "feat/my-first-change"

sh(f"git switch -c {BRANCH}", check=False)  # already exists? drop the -c
print("on branch:", sh("git branch --show-current", quiet=True))

$ git switch -c feat/my-first-change
Switched to a new branch 'feat/my-first-change'

on branch: feat/my-first-change


## 5. `add` + `commit` — save your work to git history

Make a change first (this is where your AI editor does the typing), then stage it and commit it.

Read your own diff before committing — it's the cheapest bug catch there is. Write the message as `type: summary`, like `feat: add retrieval demo` or `docs: fix setup steps`.

The cell below stages one scratch file so it's safe to run right now. In real work you'd use `git add .` to stage everything you changed.

In [5]:
from pathlib import Path

Path("scratch.md").write_text("# Scratch\n\nMy first commit on this repo.\n")

sh("git status -s")  # what changed
sh("git add scratch.md")  # stage it (real work: git add .)
sh("git --no-pager diff --staged")  # read your own diff first
sh('git commit -m "docs: add scratch file"', check=False)

$ git status -s
?? .ipynb_checkpoints/Dev_Environment_GitHub_Workflow-checkpoint.ipynb
?? Dev_Environment_GitHub_Workflow.ipynb
?? daily-git-loop.sh
?? scratch.md
?? ../02_Prompt_Engineering/
?? ../03_Agent_Harness/

$ git add scratch.md


$ git --no-pager diff --staged
diff --git a/01_First_Steps/scratch.md b/01_First_Steps/scratch.md
new file mode 100644
index 0000000..42617de
--- /dev/null
+++ b/01_First_Steps/scratch.md
@@ -0,0 +1,3 @@
+# Scratch
+
+My first commit on this repo.

$ git commit -m "docs: add scratch file"
[feat/my-first-change 18bc52f] docs: add scratch file
 1 file changed, 3 insertions(+)
 create mode 100644 01_First_Steps/scratch.md



'[feat/my-first-change 18bc52f] docs: add scratch file\n 1 file changed, 3 insertions(+)\n create mode 100644 01_First_Steps/scratch.md'

## 6. `push` — send it to your repo on GitHub

`-u` links your local branch to the one on `origin`, so after the first time a plain `git push` or `git pull` knows where to go.

In [6]:
sh(f"git push -u origin {BRANCH}", check=False)

# Working straight on main instead of a branch? That's:
# sh("git push origin main")

$ git push -u origin feat/my-first-change
branch 'feat/my-first-change' set up to track 'origin/feat/my-first-change'.
remote: 
remote: Create a pull request for 'feat/my-first-change' on GitHub by visiting:        
remote:      https://github.com/AI-Maker-Space/The-Hive-FDE-Challenge/pull/new/feat/my-first-change        
remote: 
To github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git
 * [new branch]      feat/my-first-change -> feat/my-first-change



"branch 'feat/my-first-change' set up to track 'origin/feat/my-first-change'."

## The whole loop, in one place

```bash
# once per repo
git clone git@github.com:YOUR_USERNAME/YOUR_REPO.git
cd YOUR_REPO
git remote add upstream git@github.com:AI-Maker-Space/The-Hive-FDE-Challenge.git
git remote -v

# every session
git pull upstream main              # get the latest course code
git switch -c feat/my-change        # branch for today's work
# ...make the change with Claude Code / Cursor...
git status -s                       # what changed
git add .                           # stage it
git diff --staged                   # read your own diff
git commit -m "feat: my change"     # save it to history
git push -u origin feat/my-change   # send it to your repo
```

---

## 🚀 Advanced Build

**Open a pull request without leaving the terminal.** Install the [GitHub CLI](https://cli.github.com/), run `gh auth login`, then `gh pr create --base main --fill` and `gh pr view --web`.

**Undo something.** Practice the three escape hatches on a scratch branch: `git restore <file>` (throw away uncommitted changes), `git reset --soft HEAD~1` (undo the last commit, keep the work), and `git switch -` (jump back to the previous branch).

**Protect `main`.** Add a branch protection rule on GitHub that blocks direct pushes, then try `git push origin main` and watch it get rejected. That's the safety net doing its job.